In [ ]:
import json
import csv
import numpy as np
from sklearn.svm import SVC
from sklearn.metrics import ndcg_score
from collections import defaultdict
from features import *


# Load query texts
with open("queries.txt", "r") as f:
    id_to_query = {}
    for line in f:
        qid, query = line.strip().split("\t", 1)
        id_to_query[int(qid[1:])] = query

# Load merged dataset
with open("dataset.json", "r") as f:
    dataset = json.load(f)

# Load pairwise preferences
preferences = []
with open("preferences.csv", "r", newline="") as f:
    for row in csv.DictReader(f):
        preferences.append({
            "query_id": int(row["query_id"]),
            "preferred_doc": row["preferred_doc"],
            "less_preferred_doc": row["less_preferred_doc"]
        })

# Map query IDs to their merged results and assign document IDs
documents = {}
for query, results in dataset.items():
    qid = next(k for k, v in id_to_query.items() if v == query)

    for i, doc in enumerate(results, start=1):
        doc["doc_id"] = f"D{i}"

    documents[qid] = results


def compute_features(query, doc):
    """Features from a document in the merged dataset."""
    ranks = doc.get("ranks", {})

    features = {}

    for engine in ("google", "duckduckgo"):
        rank = ranks.get(engine)

        # Rank feature: 0 if absent
        features[f"rank_{engine}"] = (
            max(0, (100 - rank) / 100) if rank is not None else 0
        )
        features[f"top1_{engine}"] = int(rank == 1)
        features[f"top10_{engine}"] = int(rank is not None and rank <= 10)
        features[f"top50_{engine}"] = int(rank is not None and rank <= 50)

    features["top1count"] = features["top1_google"] + features["top1_duckduckgo"]
    features["top10count"] = features["top10_google"] + features["top10_duckduckgo"]
    features["top50count"] = features["top50_google"] + features["top50_duckduckgo"]

    features.update(content_features(query, doc["url"], doc.get("title", "")))
    features.update(popularity_features(doc["url"], doc.get("title", "")))

    return np.array(list(features.values()), dtype=float)

X_doc = {}
for qid, results in documents.items():
    query = id_to_query[qid]
    for doc in results:
        X_doc[(qid, doc["doc_id"])] = compute_features(query, doc)


query_ids = sorted(documents)
rng = np.random.default_rng(42)
rng.shuffle(query_ids)

split = int(0.8 * len(query_ids))
train_queries = set(query_ids[:split])
test_queries = set(query_ids[split:])

X_train, y_train = [], []

for pref in preferences:
    qid = pref["query_id"]
    if qid not in train_queries:
        continue

    x_better = X_doc[(qid, pref["preferred_doc"])]
    x_worse = X_doc[(qid, pref["less_preferred_doc"])]

    X_train.append(x_better - x_worse)
    y_train.append(1)

    # Reverse pair for symmetry
    X_train.append(x_worse - x_better)
    y_train.append(-1)

X_train = np.asarray(X_train)
y_train = np.asarray(y_train)

model = SVC(kernel="linear")
model.fit(X_train, y_train)

print("Training preference pairs:", len(X_train) // 2)
print("Learned feature weights:", model.coef_[0])

ground_truth = defaultdict(dict)

with open("ground_truth.txt", "r", newline="") as f:
    for row in csv.DictReader(f):
        qid = int(row["query_id"][1:])
        ground_truth[qid][row["doc_id"]] = int(row["relevance_rank"])

ndcg_values = []

for qid in sorted(test_queries):
    docs = documents[qid]

    # Keep only documents with ground-truth labels
    judged_docs = [
        doc for doc in docs
        if doc["doc_id"] in ground_truth[qid]
    ]

    if len(judged_docs) < 2:
        continue

    doc_ids = [doc["doc_id"] for doc in judged_docs]

    # Rank documents using the trained model
    X_query = np.array([X_doc[(qid, doc_id)] for doc_id in doc_ids])
    scores = X_query @ model.coef_[0]

    # Ground-truth rank 1 is best
    true_relevance = np.array([
        len(ground_truth[qid]) - ground_truth[qid][doc_id] + 1
        for doc_id in doc_ids
    ])

    ndcg_values.append(
        ndcg_score(
            true_relevance.reshape(1, -1),
            scores.reshape(1, -1)
        )
    )

print("Test NDCG:", np.mean(ndcg_values) if ndcg_values else "No evaluable queries")

feature_names = [
    "rank_google",
    "top1_google",
    "top10_google",
    "top50_google",
    "rank_duckduckgo",
    "top1_duckduckgo",
    "top10_duckduckgo",
    "top50_duckduckgo",
    "top1count",
    "top10count",
    "top50count",
    "query_url_cosine",
    "query_title_cosine",
    "domain_name_in_query",
    "url_length",
    "abstract_contains_home",
    "url_contains_tilde"
]

with open("feature_weights.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["feature", "weight"])
    writer.writerows(zip(feature_names, model.coef_[0]))

print("Feature weights saved to feature_weights.csv")



Training preference pairs: 6713
Learned feature weights: [-35.84730586  -2.09384227   6.74691548   6.74691548 -27.76402576
   0.11540863   3.10360904   3.10360904  -1.97843364   9.85052452
   9.85052452   0.34143678  -0.25983466  -0.43989747   0.19495479
   0.           0.05536458]
Test NDCG: 0.7176938112565183
Feature weights saved to feature_weights.csv


In [5]:
# Compare RankSVM against original dataset order
baseline_ndcg = []
ranksvm_ndcg = []
baseline_pairwise = []
ranksvm_pairwise = []

for qid in sorted(test_queries):
    docs = [
        doc for doc in documents[qid]
        if doc["doc_id"] in ground_truth[qid]
    ]

    if len(docs) < 2:
        continue

    doc_ids = [doc["doc_id"] for doc in docs]

    # Ground-truth relevance: rank 1 is best
    relevance_scores = np.array([
        len(ground_truth[qid]) - ground_truth[qid][doc_id] + 1
        for doc_id in doc_ids
    ])

    # Baseline: original order in dataset.json
    # Earlier documents receive higher scores
    baseline_scores = np.array([
        -i for i in range(len(docs))
    ], dtype=float)

    # RankSVM scores
    X_query = np.array([
        X_doc[(qid, doc_id)] for doc_id in doc_ids
    ])
    ranksvm_scores = X_query @ model.coef_[0]

    baseline_ndcg.append(
        ndcg_score(
            relevance_scores.reshape(1, -1),
            baseline_scores.reshape(1, -1)
        )
    )

    ranksvm_ndcg.append(
        ndcg_score(
            relevance_scores.reshape(1, -1),
            ranksvm_scores.reshape(1, -1)
        )
    )

    # Pairwise ranking accuracy against ground truth
    # Ignore ties (if any) in the ground-truth relevance
    for i in range(len(docs)):
        for j in range(i + 1, len(docs)):
            if relevance_scores[i] == relevance_scores[j]:
                continue

            correct_order = relevance_scores[i] > relevance_scores[j]

            baseline_correct = (
                baseline_scores[i] > baseline_scores[j]
            ) == correct_order

            ranksvm_correct = (
                ranksvm_scores[i] > ranksvm_scores[j]
            ) == correct_order

            baseline_pairwise.append(baseline_correct)
            ranksvm_pairwise.append(ranksvm_correct)


print("\n--- Test-set comparison ---")
print(f"Baseline NDCG:             {np.mean(baseline_ndcg):.4f}")
print(f"RankSVM NDCG:               {np.mean(ranksvm_ndcg):.4f}")
print(f"NDCG improvement:           {np.mean(ranksvm_ndcg) - np.mean(baseline_ndcg):+.4f}")

print(f"Baseline pairwise accuracy: {np.mean(baseline_pairwise):.4f}")
print(f"RankSVM pairwise accuracy:   {np.mean(ranksvm_pairwise):.4f}")
print(f"Pairwise improvement:       {np.mean(ranksvm_pairwise) - np.mean(baseline_pairwise):+.4f}")


--- Test-set comparison ---
Baseline NDCG:             0.9173
RankSVM NDCG:               0.7177
NDCG improvement:           -0.1996
Baseline pairwise accuracy: 0.7804
RankSVM pairwise accuracy:   0.2275
Pairwise improvement:       -0.5529
